### 1. xlsx 형식으로 저장된 파일을 CSV 파일로 변환.

In [ ]:
import pandas as pd

XLSX_PATH = "./Raw_Data/CAS_KPBMA_MAP3K5_IC50s.xlsx"       # original xlsx file 
SHEET_INDEX = 1
OUT_CSV = "./Raw_Data/CAS_KPBMA_MAP3K5_IC50s_2nd_sheet.csv"       # output csv file

df = pd.read_excel(XLSX_PATH, sheet_name=SHEET_INDEX, header=1, engine="openpyxl")

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved to {OUT_CSV}")


Saved to /home/sechoi/Projects/DACON/Submission/CAS_KPBMA_MAP3K5_IC50s_2nd_sheet.csv


### 2. csv로 변환된 CAS 데이터 중, 종(Human, Homo-sapiens, Mouse)별 분석 및 확인.

In [ ]:
import numpy as np 
CAS_DF = pd.read_csv('./Raw_Data/CAS_KPBMA_MAP3K5_IC50s_2nd_sheet.csv')

# IC50: micromoloar (uM) -> nanomolar (nM) 변환
CAS_DF['IC50_nM'] = CAS_DF['Single Value (Parsed)'] * 1000

# nM로 변환된 IC50 값을 pIC50로 변환
# pIC50 = 9 - log10(IC50_nM)
CAS_DF['pIC50'] = 9 - np.log10(CAS_DF['IC50_nM'])

CAS_DF.head()


In [ ]:
species_data = {}

for species_name in ["HOMO SAPIENS", "Human", "Mouse"]:
    species_mask = CAS_DF["Species"].str.upper() == species_name.upper()
    species_df = CAS_DF[species_mask]
    
    # Single Value (Parse) 열에서 NaN 값이 아닌 데이터 선택 
    species_df_filtered = species_df[species_df["pIC50"].notna()]
    
    if len(species_df_filtered) > 0:
        species_data[species_name] = {
            "count": len(species_df_filtered),
            "pIC50_min": species_df_filtered["pIC50"].min(),
            "pIC50_max": species_df_filtered["pIC50"].max(),
            "pIC50_mean": species_df_filtered["pIC50"].mean()
        }


# Species별 NaN 개수 확인, 단 Single Value (Parsed)가 있는 경우만
nan_count = CAS_DF[CAS_DF["pIC50"].notna()]["Species"].isna().sum()

print("Species별 개수와 pIC50 범위:")
for species, data in species_data.items():
    print(f"  {species}: {data['count']}개")
    print(f"    pIC50 범위: {data['pIC50_min']:.6f} ~ {data['pIC50_max']:.6f}")
    print(f"   pIC50 평균: {data['pIC50_mean']:.6f}")
    print()

print(f"  NaN Species (Single Value 있는 경우): {nan_count}개")

# 전체 Single Value (Parsed) 범위 (NaN 제외)
single_value_min = CAS_DF["pIC50"].min()
single_value_max = CAS_DF["pIC50"].max()
print(f"\n전체 pIC50 값 범위: {single_value_min:.6f} ~ {single_value_max:.6f}")

Species별 개수와 pIC50 범위:
  HOMO SAPIENS: 335개
    pIC50 범위: 4.200659 ~ 9.397940
   pIC50 평균: 6.783605

  Human: 532개
    pIC50 범위: 6.000000 ~ 13.000000
   pIC50 평균: 7.421284

  Mouse: 2개
    pIC50 범위: 4.829738 ~ 4.974694
   pIC50 평균: 4.902216

  NaN Species (Single Value 있는 경우): 2178개

전체 pIC50 값 범위: 3.301030 ~ 13.000000


### 3. CAS 데이터 분석 이후, 필터링 진행.

In [ ]:
Clean_CAS_df = CAS_DF.copy()

# Species가 HOMO SAPIENS와 Human인 데이터만 필터링
species_mask = (
    Clean_CAS_df['Species'].str.upper().isin(['HOMO SAPIENS', 'HUMAN']) | 
    Clean_CAS_df['Species'].isna()
)

filtered_cas_df = Clean_CAS_df[species_mask].copy()

# pIC50 값이 존재하는 데이터만 필터링
cas_final_df = filtered_cas_df[filtered_cas_df['pIC50'].notna()][['SMILES', 'pIC50']].copy()

# 컬럼 이름 변경
cas_final_df = cas_final_df.rename(columns={
    'SMILES': 'Smiles'
})

# 결과 확인
print("필터링 결과:")
print(f"  원본 데이터 개수: {len(Clean_CAS_df)}개")
print(f"  HOMO SAPIENS + Human 데이터 개수: {len(filtered_cas_df)}개")
print(f"  pIC50 값이 존재하는 데이터 개수: {len(cas_final_df)}개")

print("\nSpecies별 분포:")
species_counts = filtered_cas_df['Species'].value_counts()
for species, count in species_counts.items():
    print(f"  {species}: {count}개")

print("\n최종 데이터 샘플:")
print(cas_final_df.head(10))

# pIC50 값의 통계 정보
print(f"\npIC50 값 통계:")
print(f"  최솟값: {cas_final_df['pIC50'].min():.4f}")
print(f"  최댓값: {cas_final_df['pIC50'].max():.4f}")
print(f"  평균값: {cas_final_df['pIC50'].mean():.4f}")

# CSV로 저장장
cas_final_df.to_csv('./Raw_Data/CAS_clean_SP_pIC50.csv', index=False)



필터링 결과:
  원본 데이터 개수: 3452개
  HOMO SAPIENS + Human 데이터 개수: 3450개
  pIC50 값이 존재하는 데이터 개수: 3045개

Species별 분포:
  Human: 658개
  HOMO SAPIENS: 509개

최종 데이터 샘플:
                                               Smiles     pIC50
0   O[C@H]1[C@H](N2C=3C(N=C2)=C(N)N=CN3)O[C@H](COP...  5.613799
1                          S(C)C1=C2C(=NC(N)=N1)N=CN2  4.876148
2                   N(C1=C2C(N=CN2)=NC=N1)C3=CC=CC=C3  5.000000
3                         C12=C(N=CN=C1N=CN2)N3CCCCC3  4.769551
4    C(C)(=O)C1=C2C=3C(C(=O)C=4C2=CC=CC4)=CC=CC3NC1=O  4.823909
5               N(C1=C2C(N=CN2)=NC=N1)C3=CC(Cl)=CC=C3  5.000000
6                      N(CCC)(CCC)C1=C2C(N=CN2)=NC=N1  4.588380
7                 N(C1=NC2=C(C=N1)C=CC=C2)C3=CC=CC=C3  5.000000
10  C[C@]12N3C=4C5=C(C6=C(C4C=7C3=CC=CC7)CNC6=O)C=...  7.612610
11  C[C@]12N3C=4C5=C(C6=C(C4C=7C3=CC=CC7)CNC6=O)C=...  7.642065

pIC50 값 통계:
  최솟값: 3.3010
  최댓값: 13.0000
  평균값: 7.2742


### 4. 제공된 ChEMBL 데이터 분석.

In [ ]:
Chembl_df = pd.read_csv('./Raw_Data/ChEMBL_ASK1(IC50).csv', sep=';')
# BAO Label별로 pCHEMBL column 개수와 NaN 개수 확인
bao_counts = {}

# 각 BAO Label별로 개수와 NaN 개수 계산
bao_labels = ['single protein format', 'assay format', 'cell-based format']

for label in bao_labels:
    mask = Chembl_df['BAO Label'] == label
    label_df = Chembl_df[mask]
    
    total_count = label_df.shape[0]
    nan_count = label_df['pChEMBL Value'].isna().sum()
    valid_count = total_count - nan_count
    
    bao_counts[label] = {
        'total': total_count,
        'nan': nan_count,
        'valid': valid_count
    }

# 결과 출력
print("BAO Label별 개수와 pCHEMBL NaN 개수:")
for label, data in bao_counts.items():
    print(f"  {label}:")
    print(f"    전체: {data['total']}개")
    print(f"    pCHEMBL NaN: {data['nan']}개")
    print(f"    pCHEMBL 유효값: {data['valid']}개")
    print()

# 전체 개수도 확인
total_count = Chembl_df.shape[0]
total_nan = Chembl_df['pChEMBL Value'].isna().sum()
total_valid = total_count - total_nan

print(f"전체 데이터:")
print(f"  전체: {total_count}개")
print(f"  pCHEMBL NaN: {total_nan}개")
print(f"  pCHEMBL 유효값: {total_valid}개")

# 비율도 계산
print(f"\n전체 pCHEMBL NaN 비율: {(total_nan/total_count)*100:.1f}%")
#Chembl_single_protein_df.to_csv('/home/sechoi/Projects/DACON/Summary/Chembl_single_protein_df.csv', index=False)

BAO Label별 개수와 pCHEMBL NaN 개수:
  single protein format:
    전체: 414개
    pCHEMBL NaN: 61개
    pCHEMBL 유효값: 353개

  assay format:
    전체: 50개
    pCHEMBL NaN: 34개
    pCHEMBL 유효값: 16개

  cell-based format:
    전체: 360개
    pCHEMBL NaN: 15개
    pCHEMBL 유효값: 345개

전체 데이터:
  전체: 824개
  pCHEMBL NaN: 110개
  pCHEMBL 유효값: 714개

전체 pCHEMBL NaN 비율: 13.3%


### 5. CheMBL 데이터 분석 이후, 필터링 진행.

In [ ]:
# assay format 제외하고 single protein format과 cell-based format만 사용
format_mask = Chembl_df['BAO Label'].isin(['single protein format', 'cell-based format'])
chembl_filtered_df = Chembl_df[format_mask].copy()

# pChEMBL Value에서 NaN값 제외
chembl_clean_df = chembl_filtered_df[chembl_filtered_df['pChEMBL Value'].notna()].copy()

chembl_final_df = chembl_clean_df[['Smiles', 'pChEMBL Value']].copy()
chembl_final_df = chembl_final_df.rename(columns={'pChEMBL Value': 'pIC50'})

# 결과 확인
print(f"원본 데이터 개수: {len(Chembl_df)}")
print(f"BAO Label 필터링 후 개수: {len(chembl_filtered_df)}")
print(f"pCHEMBL Value NaN 제외 후 개수: {len(chembl_clean_df)}")
print(f"최종 데이터 개수: {len(chembl_final_df)}")

# BAO Label별 분포 확인
print(f"\nBAO Label별 분포:")
bao_counts = chembl_filtered_df['BAO Label'].value_counts()
for label, count in bao_counts.items():
    print(f"  {label}: {count}개")

print("\n최종 데이터 샘플:")
print(chembl_final_df.head())

print(f"\npIC50 값 범위: {chembl_final_df['pIC50'].min():.4f} ~ {chembl_final_df['pIC50'].max():.4f}")
print(f"pIC50 평균: {chembl_final_df['pIC50'].mean():.4f}")

# 필요시 CSV로 저장
chembl_final_df.to_csv('./Raw_Data/chembl_clean_noassay_pIC50.csv', index=False)

원본 데이터 개수: 824
BAO Label 필터링 후 개수: 774
pCHEMBL Value NaN 제외 후 개수: 698
최종 데이터 개수: 698

BAO Label별 분포:
  single protein format: 414개
  cell-based format: 360개

최종 데이터 샘플:
                                              Smiles  pIC50
0   Cn1cc(Cl)c2cnc(NC(=O)c3ccc([C@](C)(O)CO)cc3)cc21   7.42
1         Cc1cc2c(-c3ccc(S(=O)(=O)NCCN)s3)ccnc2[nH]1   6.60
2  Cc1cc(C)c2nc(N3C(=O)C(O)=C(C(=O)c4ccc(Cl)cc4)C...   5.20
3  Cc1ccc(C(=O)C2=C(O)C(=O)N(c3nc4ccc(F)cc4s3)C2c...   5.12
4  CCOc1ccc2nc(N3C(=O)C(O)=C(C(=O)c4ccccc4)C3c3cc...   5.38

pIC50 값 범위: 4.0000 ~ 9.1600
pIC50 평균: 6.8315


### 6. 제공된 Pubchem 데이터 분석.

In [ ]:
Pubchem_df = pd.read_csv('./Raw_Data/Pubchem_ASK1.csv', sep=',')

# Activity column별로 Activity_Value의 range와 NaN 개수 확인
activity_counts = {}

# 각 Activity 값별로 개수, Activity_Value range, NaN 개수 계산
activity_values = ['Active', 'Inactive', 'Inconclusive', 'Unspecified']

for activity in activity_values:
    mask = Pubchem_df['Activity'] == activity
    activity_df = Pubchem_df[mask]
    
    total_count = activity_df.shape[0]
    nan_count = activity_df['Activity_Value'].isna().sum()
    valid_count = total_count - nan_count
    
    # Activity_Value의 range 계산 (NaN 제외)
    if valid_count > 0:
        min_val = activity_df['Activity_Value'].min()
        max_val = activity_df['Activity_Value'].max()
        mean_val = activity_df['Activity_Value'].mean()
    else:
        min_val = max_val = mean_val = None
    
    activity_counts[activity] = {
        'total': total_count,
        'nan': nan_count,
        'valid': valid_count,
        'min': min_val,
        'max': max_val,
        'mean': mean_val
    }

# 결과 출력
print("Activity별 개수와 Activity_Value 정보:")
for activity, data in activity_counts.items():
    print(f"  {activity}:")
    print(f"    전체: {data['total']}개")
    print(f"    Activity_Value NaN: {data['nan']}개")
    print(f"    Activity_Value 유효값: {data['valid']}개")
    
    if data['valid'] > 0:
        print(f"    Activity_Value 범위: {data['min']:.6f} ~ {data['max']:.6f}")
        print(f"    Activity_Value 평균: {data['mean']:.6f}")
    else:
        print(f"    Activity_Value 범위: 데이터 없음")
        print(f"    Activity_Value 평균: 데이터 없음")
    print()

# 전체 개수도 확인
total_count = Pubchem_df.shape[0]
total_nan = Pubchem_df['Activity_Value'].isna().sum()
total_valid = total_count - total_nan

print(f"전체 데이터:")
print(f"  전체: {total_count}개")
print(f"  Activity_Value NaN: {total_nan}개")
print(f"  Activity_Value 유효값: {total_valid}개")

# 비율도 계산
print(f"\n전체 Activity_Value NaN 비율: {(total_nan/total_count)*100:.1f}%")


Activity별 개수와 Activity_Value 정보:
  Active:
    전체: 953개
    Activity_Value NaN: 274개
    Activity_Value 유효값: 679개
    Activity_Value 범위: 0.000100 ~ 10.000000
    Activity_Value 평균: 1.609882

  Inactive:
    전체: 22071개
    Activity_Value NaN: 21730개
    Activity_Value 유효값: 341개
    Activity_Value 범위: 10.218000 ~ 27.499000
    Activity_Value 평균: 18.398062

  Inconclusive:
    전체: 4개
    Activity_Value NaN: 4개
    Activity_Value 유효값: 0개
    Activity_Value 범위: 데이터 없음
    Activity_Value 평균: 데이터 없음

  Unspecified:
    전체: 767개
    Activity_Value NaN: 639개
    Activity_Value 유효값: 128개
    Activity_Value 범위: 0.100000 ~ 500.000000
    Activity_Value 평균: 20.612758

전체 데이터:
  전체: 23795개
  Activity_Value NaN: 22647개
  Activity_Value 유효값: 1148개

전체 Activity_Value NaN 비율: 95.2%


/tmp/ipykernel_1753413/1818652906.py:2: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  Pubchem_df = pd.read_csv('/home/sechoi/Projects/DACON/BOB/Pubchem_ASK1.csv', sep=',')


### 7. PubChem 데이터 분석 이후, 필터링 진행.

In [ ]:
# 조건 1: Activity가 Active이면서 Activity_Type이 IC50인 데이터
condition_active = (Pubchem_df['Activity'] == 'Active') & (Pubchem_df['Activity_Type'] == 'IC50')
active_ic50_df = Pubchem_df[condition_active].copy()

# 조건 2: Activity가 Unspecified이면서 Activity_Type이 IC50이고 Activity_Value가 10보다 작은 데이터
condition_unspecified = (
    (Pubchem_df['Activity'] == 'Unspecified') & 
    (Pubchem_df['Activity_Type'] == 'IC50') & 
    (Pubchem_df['Activity_Value'] < 10)
)
unspecified_ic50_df = Pubchem_df[condition_unspecified].copy()

# 두 조건의 데이터를 합치기
filtered_pubchem_df = pd.concat([active_ic50_df, unspecified_ic50_df], ignore_index=True)

# 결과 확인
print("필터링 결과:")
print(f"  Active + IC50: {len(active_ic50_df)}개")
print(f"  Unspecified + IC50 + Activity_Value < 10: {len(unspecified_ic50_df)}개")
print(f"  총 필터링된 데이터: {len(filtered_pubchem_df)}개")

print(f"\n원본 데이터 개수: {len(Pubchem_df)}개")
print(f"필터링 후 데이터 개수: {len(filtered_pubchem_df)}개")

# Activity별 분포 확인
print(f"\n필터링된 데이터의 Activity 분포:")
activity_dist = filtered_pubchem_df['Activity'].value_counts()
for activity, count in activity_dist.items():
    print(f"  {activity}: {count}개")

# 필요한 컬럼만 선택하고 이름 변경
pubchem_final_df = filtered_pubchem_df[['SMILES', 'Activity_Value']].copy()

# 컬럼 이름 변경
pubchem_final_df = pubchem_final_df.rename(columns={'SMILES': 'Smiles'})

# Activity_Value를 변환하여 pIC50으로 이름 변경
# Activity_Value * 1000 (uM -> nM) -> 9 - log10(IC50_nM)
pubchem_final_df['pIC50'] = 9 - np.log10(pubchem_final_df['Activity_Value'] * 1000)

# 결과 확인
print("최종 데이터 구조:")
print(f"  컬럼: {list(pubchem_final_df.columns)}")
print(f"  데이터 개수: {len(pubchem_final_df)}개")

print("\n최종 데이터 샘플:")
print(pubchem_final_df.head(10))

# pIC50 값의 통계 정보
print(f"\npIC50 값 통계:")
print(f"  최솟값: {pubchem_final_df['pIC50'].min():.4f}")
print(f"  최댓값: {pubchem_final_df['pIC50'].max():.4f}")
print(f"  평균값: {pubchem_final_df['pIC50'].mean():.4f}")
print(f"  표준편차: {pubchem_final_df['pIC50'].std():.4f}")

# CSV로 저장
pubchem_final_df[['Smiles', 'pIC50']].to_csv('./Raw_Data/Pubchem_clean_pIC50.csv', index=False)


필터링 결과:
  Active + IC50: 564개
  Unspecified + IC50 + Activity_Value < 10: 31개
  총 필터링된 데이터: 595개

원본 데이터 개수: 23795개
필터링 후 데이터 개수: 595개

필터링된 데이터의 Activity 분포:
  Active: 564개
  Unspecified: 31개


### 8. Python Script를 활용하여, CheMBLE API를 통하여 추가 데이터 다운로드 

In [ ]:
"""
ChEMBL에서 ASK1/MAP3K5 추가 데이터 다운로드
"""

import requests
import pandas as pd
import json
import time
from pathlib import Path

def download_chembl_data(target_id, target_name):
    """ChEMBL에서 특정 타겟의 데이터를 다운로드"""
    
    print(f"=== {target_name} ({target_id}) 데이터 다운로드 ===")
    
    # IC50 데이터 다운로드
    url = f"https://www.ebi.ac.uk/chembl/api/data/activity.json?target_chembl_id={target_id}&standard_type=IC50"
    
    try:
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            activities = data.get('activities', [])
            
            print(f"다운로드된 데이터: {len(activities)} 개")
            
            if activities:
                # 데이터프레임으로 변환
                df = pd.DataFrame(activities)
                
                # 필요한 컬럼만 선택
                columns_to_keep = [
                    'canonical_smiles', 'standard_value', 'standard_units', 
                    'standard_relation', 'target_pref_name', 'molecule_pref_name',
                    'document_journal', 'document_year', 'assay_description'
                ]
                
                available_columns = [col for col in columns_to_keep if col in df.columns]
                df_clean = df[available_columns].copy()
                
                # 컬럼명 정규화
                df_clean = df_clean.rename(columns={
                    'canonical_smiles': 'Smiles',
                    'standard_value': 'Standard_Value',
                    'standard_units': 'Standard_Units',
                    'standard_relation': 'Standard_Relation',
                    'molecule_pref_name': 'Compound_Name',
                    'document_journal': 'Journal',
                    'document_year': 'Year',
                    'assay_description': 'Assay_Description'
                    'target_pref_name': 'Target_Name',
                })
                
                # 데이터 정제
                df_clean['Standard_Value'] = pd.to_numeric(df_clean['Standard_Value'], errors='coerce')
                df_clean = df_clean.dropna(subset=['Standard_Value', 'Smiles'])
                
                print(f"정제된 데이터: {len(df_clean)} 개")
                print(f"IC50 범위: {df_clean['Standard_Value'].min():.2f} ~ {df_clean['Standard_Value'].max():.2f}")
                
                # 단위 분포 확인
                if 'Standard_Units' in df_clean.columns:
                    unit_counts = df_clean['Standard_Units'].value_counts()
                    print(f"단위 분포:")
                    for unit, count in unit_counts.items():
                        print(f"  {unit}: {count} 개")
                
                return df_clean
            else:
                print("데이터가 없습니다.")
                return None
                
        else:
            print(f"API 호출 실패: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"다운로드 오류: {e}")
        return None

def process_additional_data():
    """추가 데이터 처리 및 저장"""
    
    print("=== ChEMBL 추가 데이터 처리 ===")
    
    # 추가 타겟 정보
    additional_targets = [
        {'id': 'CHEMBL1075138', 'name': 'ASK1'},
        {'id': 'CHEMBL1075140', 'name': 'MAP3K5/ASK1'}
    ]
    
    all_additional_data = []
    
    for target in additional_targets:
        df = download_chembl_data(target['id'], target['name'])
        
        if df is not None:
            df['source'] = f"ChEMBL_{target['name']}"
            all_additional_data.append(df)
        
        time.sleep(1)  # API 호출 제한 방지
    
    if all_additional_data:
        # 모든 추가 데이터 통합
        combined_df = pd.concat(all_additional_data, ignore_index=True)
        
        print(f"\n=== 통합된 추가 데이터 요약 ===")
        print(f"총 데이터: {len(combined_df)} 개")
        print(f"고유 SMILES: {len(combined_df['Smiles'].unique())} 개")
        
        # 소스별 분포
        source_counts = combined_df['source'].value_counts()
        print(f"\n소스별 분포:")
        for source, count in source_counts.items():
            print(f"  {source}: {count} 개")
        
        # IC50 값 분포
        print(f"\nIC50 값 분포:")
        print(f"  최소값: {combined_df['Standard_Value'].min():.2f}")
        print(f"  최대값: {combined_df['Standard_Value'].max():.2f}")
        print(f"  평균: {combined_df['Standard_Value'].mean():.2f}")
        print(f"  중간값: {combined_df['Standard_Value'].median():.2f}")
        
        # 단위 변환 (nM로 통일)
        combined_df['IC50_nM'] = combined_df['Standard_Value'].copy()
        
        if 'Standard_Units' in combined_df.columns:
            # μM를 nM로 변환
            mu_to_nm = combined_df['Standard_Units'].str.contains('μM|uM|microM', case=False, na=False)
            combined_df.loc[mu_to_nm, 'IC50_nM'] = combined_df.loc[mu_to_nm, 'Standard_Value'] * 1000
            
            # mM를 nM로 변환
            mm_to_nm = combined_df['Standard_Units'].str.contains('mM|milliM', case=False, na=False)
            combined_df.loc[mm_to_nm, 'IC50_nM'] = combined_df.loc[mm_to_nm, 'Standard_Value'] * 1000000
            
            # pM를 nM로 변환
            pm_to_nm = combined_df['Standard_Units'].str.contains('pM|picoM', case=False, na=False)
            combined_df.loc[pm_to_nm, 'IC50_nM'] = combined_df.loc[pm_to_nm, 'Standard_Value'] * 0.001
        
        print(f"\n변환 후 IC50 범위: {combined_df['IC50_nM'].min():.2f} ~ {combined_df['IC50_nM'].max():.2f} nM")
        
        # 저장
        output_file = "./Raw_Data/additional_chembl_ask1_data.csv"
        Path("raw_data").mkdir(exist_ok=True)
        
        combined_df.to_csv(output_file, index=False)
        print(f"\n✅ 추가 데이터가 '{output_file}'에 저장되었습니다.")
        
        return combined_df
    else:
        print("다운로드할 수 있는 추가 데이터가 없습니다.")
        return None

def main():
    """메인 함수"""
    
    print("ChEMBL 추가 ASK1/MAP3K5 데이터 다운로드 및 저장장")
    print("="*60)
    
    # 추가 데이터 다운로드
    process_additional_data()


if __name__ == "__main__":
    main()

### 9. SMILES 데이터 전처리 및 데이터 프레임 통합

In [ ]:
from rdkit import Chem
import re

def remove_cl_salts(smiles):
    """
    SMILES에서 .cl, .cl.cl, .Cl, .Cl.Cl 등을 제거하는 함수
    """
    if pd.isna(smiles) or smiles is None:
        return None
    
    # .cl, .cl.cl, .Cl, .Cl.Cl 등 제거 (대소문자 구분 없이)
    cleaned_smiles = re.sub(r'\.cl\.cl\.?', '', smiles, flags=re.IGNORECASE)  # .cl.cl 또는 .cl.cl.
    cleaned_smiles = re.sub(r'\.cl\.?', '', cleaned_smiles, flags=re.IGNORECASE)  # .cl 또는 .cl.
    
    # 추가로 .Cl, .Cl.Cl 등도 제거
    cleaned_smiles = re.sub(r'\.Cl\.Cl\.?', '', cleaned_smiles)  # .Cl.Cl 또는 .Cl.Cl.
    cleaned_smiles = re.sub(r'\.Cl\.?', '', cleaned_smiles)  # .Cl 또는 .Cl.
    
    # 연속된 점(.) 제거 (예: .. -> .)
    cleaned_smiles = re.sub(r'\.+', '.', cleaned_smiles)
    
    # 시작과 끝의 점(.) 제거
    cleaned_smiles = cleaned_smiles.strip('.')
    
    return cleaned_smiles if cleaned_smiles else None

def clean_and_canonicalize(smiles):
    """
    SMILES에서 .cl salt를 제거하고 canonicalize하는 함수
    """
    # 1단계: .cl salt 제거
    cleaned_smiles = remove_cl_salts(smiles)
    
    if cleaned_smiles is None:
        return None
    
    # 2단계: canonicalize
    try:
        mol = Chem.MolFromSmiles(cleaned_smiles)
        if mol is None:
            return None
        
        # 가장 큰 fragment만 추출 (counter-ion 제거)
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
        if not frags:
            return None
            
        largest_frag = max(frags, key=lambda m: m.GetNumAtoms())

        # Canonical SMILES 변환
        return Chem.MolToSmiles(largest_frag, canonical=True)
    
    except:
        return None

# CAS 데이터에 적용
cleaned_cas_df = pd.read_csv('./Raw_Data/CAS_clean_SP_pIC50.csv')
cleaned_chembl_df = pd.read_csv('./Raw_Data/chembl_clean_noassay_pIC50.csv')
cleaned_pubchem_df = pd.read_csv('./Raw_Data/Pubchem_clean_pIC50.csv')
add_chembl_df = pd.read_csv('./Raw_Data/additional_chembl_ask1_data.csv')

# 각 데이터프레임에 Canonical_Smiles 컬럼 추가
cleaned_cas_df['Canonical_Smiles'] = cleaned_cas_df['Smiles'].apply(clean_and_canonicalize)
cleaned_chembl_df['Canonical_Smiles'] = cleaned_chembl_df['Smiles'].apply(clean_and_canonicalize)
cleaned_pubchem_df['Canonical_Smiles'] = cleaned_pubchem_df['Smiles'].apply(clean_and_canonicalize)
add_chembl_df['Canonical_Smiles'] = add_chembl_df['Smiles'].apply(clean_and_canonicalize)
add_chembl_df['pIC50'] = -np.log10(add_chembl_df['Standard_Value']*1e-9)

# 최종 데이터프레임 생성
train_df = pd.concat([cleaned_cas_df, cleaned_chembl_df, cleaned_pubchem_df, add_chembl_df], ignore_index=True)
train_df_final = train_df[['Canonical_Smiles', 'pIC50']].copy()
train_df_final.to_csv('./Raw_Data/train_df.csv', index=False)

### 10. 통합 데이터 중, Canonical SMILES 기준, 중복 되는 pIC50값 존재 시, Max 또는 Min 값을 기준으로 데이터 분류 및 저장.

In [ ]:
# pIC50 NaN 값 제거
train_df_clean = train_df_final.dropna(subset=['pIC50']).copy()

print(f"NaN 제거 후 데이터 개수: {len(train_df_clean)}개")

# Canonical_Smiles 기준으로 중복 확인
duplicate_counts = train_df_clean['Canonical_Smiles'].value_counts()
duplicate_smiles = duplicate_counts[duplicate_counts > 1]

print(f"\n중복된 SMILES 개수: {len(duplicate_smiles)}개")

# 중복된 SMILES에 대해 pIC50 최댓값과 최솟값 계산
max_pic50_df = train_df_clean.groupby('Canonical_Smiles')['pIC50'].max().reset_index()
min_pic50_df = train_df_clean.groupby('Canonical_Smiles')['pIC50'].min().reset_index()

# 컬럼명 변경
max_pic50_df = max_pic50_df.rename(columns={'pIC50': 'pIC50'})
min_pic50_df = min_pic50_df.rename(columns={'pIC50': 'pIC50'})

# CSV로 저장
max_pic50_df.to_csv('./Raw_Data/train_df_final_max.csv', index=False)
min_pic50_df.to_csv('./Raw_Data/train_df_final_min.csv', index=False)

NaN 제거 후 데이터 개수: 4366개

중복된 SMILES 개수: 830개
